# 引入所需package

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

# 搭建简单模型

In [5]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784,128,bias=True)
        self.relu=nn.ReLU()

    def forward(self,x):
        x=x.view(-1,784)
        x=self.fc1(x)
        x=self.relu(x)
        return x

# 训练

In [6]:
def train():
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,),(0.3081,))
    ])

    train_dataset=datasets.MNIST('./data',train=True, download=True,transform=transform)
    train_loader=torch.utils.data.DataLoader(train_dataset,batch_size=64,shuffle=True)

    model=SimpleMLP().to(device)
    optimizer=optim.Adam(model.parameters(),lr=0.001)
    criterion=nn.MSELoss()

    for epoch in range(10):
        for batch_idx, (data,target) in enumerate(train_loader):
            data=data.to(device)
            optimizer.zero_grad()
            output=model(data)

            loss=output.pow(2).mean()
            loss.backward()
            optimizer.step()

            if batch_idx %100 ==0:
                print(f'Epoch {epoch},Batch {batch_idx}, Loss:{loss.item():.4f}')


    torch.save(model.state_dict(), 'simple_mlp.pth')
    return model

In [7]:
if __name__=='__main__':
    model=train()


Epoch 0,Batch 0, Loss:0.1639
Epoch 0,Batch 100, Loss:0.0000
Epoch 0,Batch 200, Loss:0.0000
Epoch 0,Batch 300, Loss:0.0000
Epoch 0,Batch 400, Loss:0.0000
Epoch 0,Batch 500, Loss:0.0000
Epoch 0,Batch 600, Loss:0.0000
Epoch 0,Batch 700, Loss:0.0000
Epoch 0,Batch 800, Loss:0.0000
Epoch 0,Batch 900, Loss:0.0000
Epoch 1,Batch 0, Loss:0.0000
Epoch 1,Batch 100, Loss:0.0000
Epoch 1,Batch 200, Loss:0.0000
Epoch 1,Batch 300, Loss:0.0000
Epoch 1,Batch 400, Loss:0.0000
Epoch 1,Batch 500, Loss:0.0000
Epoch 1,Batch 600, Loss:0.0000
Epoch 1,Batch 700, Loss:0.0000
Epoch 1,Batch 800, Loss:0.0000
Epoch 1,Batch 900, Loss:0.0000
Epoch 2,Batch 0, Loss:0.0000
Epoch 2,Batch 100, Loss:0.0000
Epoch 2,Batch 200, Loss:0.0000
Epoch 2,Batch 300, Loss:0.0000
Epoch 2,Batch 400, Loss:0.0000
Epoch 2,Batch 500, Loss:0.0000
Epoch 2,Batch 600, Loss:0.0000
Epoch 2,Batch 700, Loss:0.0000
Epoch 2,Batch 800, Loss:0.0000
Epoch 2,Batch 900, Loss:0.0000
Epoch 3,Batch 0, Loss:0.0000
Epoch 3,Batch 100, Loss:0.0000
Epoch 3,Batch 20

# 量化函数

In [18]:
def quantize_tensor(tensor, bits=8):
    qmin = -(2 ** (bits - 1))
    qmax = 2 ** (bits - 1) - 1

    min_val = tensor.min().item()
    max_val = tensor.max().item()


    diff = max_val - min_val
    
    if diff == 0:

        scale = 1.0
        zero_point = qmin - min_val
    else:
        scale = diff / (qmax - qmin)
        zero_point = qmin - min_val / scale


    q_tensor = torch.clamp(torch.round(tensor / scale + zero_point), qmin, qmax)
    
    return q_tensor.to(torch.int8), scale, zero_point


# 导出量化信息

In [19]:
import os
def export_weights_to_hex(model_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    """导出权重为Verilog可读的hex文件"""
    model = SimpleMLP()
    model.load_state_dict(torch.load(model_path))
    
    # 提取第一层权重和偏置
    weight = model.fc1.weight.data  # [128, 784]
    bias = model.fc1.bias.data      # [128]
    
    # 量化
    q_weight, w_scale, w_zero = quantize_tensor(weight)
    q_bias, b_scale, b_zero = quantize_tensor(bias)
    
    # 保存为hex格式（每行一个INT8值）
    with open(f'{output_dir}/weights.hex', 'w') as f:
        for row in q_weight:
            for val in row:
                f.write(f'{val.item() & 0xFF:02x}\n')
    
    with open(f'{output_dir}/bias.hex', 'w') as f:
        for val in q_bias:
            f.write(f'{val.item() & 0xFF:02x}\n')
    
    # 保存scale信息（用于反量化验证）
    np.savez(f'{output_dir}/quant_params.npz',
             w_scale=w_scale, w_zero=w_zero,
             b_scale=b_scale, b_zero=b_zero)
    
    print(f"Weight shape: {q_weight.shape}")
    print(f"Weight scale: {w_scale}, zero_point: {w_zero}")
    print(f"Bias scale: {b_scale}, zero_point: {b_zero}")


# 执行并获取量化信息

In [20]:
if __name__ == '__main__':
    export_weights_to_hex('simple_mlp.pth', 'quantized_data')

Weight shape: torch.Size([128, 784])
Weight scale: 0.0004007391631603241, zero_point: 6.275233656154057
Bias scale: 0.0002741283937996509, zero_point: 29.838008078500252


# 生成测试向量


In [21]:
def generate_test_vectors(num_samples=10):
    """生成测试输入和golden输出"""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    
    # 加载量化后的模型
    model = SimpleMLP()
    model.load_state_dict(torch.load('simple_mlp.pth'))
    model.eval()
    
    # 加载量化参数
    quant_params = np.load('quantized_data/quant_params.npz')
    
    for i in range(num_samples):
        img, label = test_dataset[i]
        img_flat = img.view(-1, 784)
        
        # 量化输入
        q_img, img_scale, img_zero = quantize_tensor(img_flat)
        
        # 软件golden结果
        with torch.no_grad():
            output = model(img_flat)
        
        # 保存输入
        with open(f'quantized_data/input_{i}.hex', 'w') as f:
            for val in q_img.flatten():
                f.write(f'{val.item() & 0xFF:02x}\n')
        
        # 保存golden输出
        q_output, _, _ = quantize_tensor(output)
        with open(f'quantized_data/golden_{i}.hex', 'w') as f:
            for val in q_output.flatten():
                f.write(f'{val.item() & 0xFF:02x}\n')
        
        print(f"Generated test vector {i}, label: {label}")



In [22]:
if __name__ == '__main__':
    generate_test_vectors()

Generated test vector 0, label: 7
Generated test vector 1, label: 2
Generated test vector 2, label: 1
Generated test vector 3, label: 0
Generated test vector 4, label: 4
Generated test vector 5, label: 1
Generated test vector 6, label: 4
Generated test vector 7, label: 9
Generated test vector 8, label: 5
Generated test vector 9, label: 9
